# Klassifikation mit verschiedenen Leveln der Personalisierung

1. Subject-independent
2. Subject Baseline Norm - t0 als baseline; t0 mittel wird von allen samples abgezogen
3. Mit Kalibrierungs Samples - t0-t5; 2,4,6 samples pro stufe mit ins training
4. Nur auf Daten einer Person - cv within subject

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from scripts.myml import loso_binary_nested_cv, loso_multiclass_nested_cv, loso_binary_calibrated_nested_cv, safe_results_binary
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from scripts.feature_engeneering import subject_baseline_normalization
from xgboost import XGBClassifier

### Daten Laden

In [4]:
DATASET_PATH = Path("dataset/np-dataset")
X = np.load(DATASET_PATH / "X.npy")
X_catch22 = np.load(DATASET_PATH / "X_catch22.npy")
X_catch22_feature_names = np.load(DATASET_PATH / "feature_names.npy")
y_heater = np.load(DATASET_PATH / "y_heater.npy")
subjects = np.load(DATASET_PATH / "subjects.npy")
y = np.argmax(y_heater, axis=1)

In [5]:
print(f"X shape : {X_catch22.shape}")
print(f"y shape : {y.shape}   classes: {np.unique(y).tolist()}")
print(f"subjects : {np.unique(subjects).shape[0]} unique subjects")
print(f"Class counts : { {c: int((y==c).sum()) for c in range(6)} }")

X shape : (2495, 174)
y shape : (2495,)   classes: [0, 1, 2, 3, 4, 5]
subjects : 52 unique subjects
Class counts : {0: 416, 1: 416, 2: 416, 3: 415, 4: 416, 5: 416}


### Hyperparameter Grid

In [ ]:
# Random Forest
PARAM_GRID_RF = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

PARAM_GRID_XGB = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
}

INNER_N_SPLITS = 5
RANDOM_STATE = 42

## For Comparison

In [ ]:
comparison_results = []

# 1. Subject-Independent
und
# 2. Subject-Baseline-Norm 

In [7]:
np.isnan(X_catch22).sum()

np.int64(1029)

In [8]:
# without normalization
X_catch22 = X_catch22.copy()

# with subject-specific normalization
baseline_class = 0 # samples with no stimulus 
X_catch22_normalized = subject_baseline_normalization(X_catch22, y, subjects, baseline_class)

print(np.nanmean(X_catch22))
print(np.nanmean(X_catch22_normalized))

40.75209365022363
1.2530141390178788


## Binary Klassifikation t0 vs t5 - Random Forest Classifier

In [ ]:
# Define comparison
class_comparison = (0, 5)
pers_comparison = [X_catch22, X_catch22_normalized]

# Train and evaluate
for X_comp in pers_comparison:

    print(f"Personalization: {'Baseline Normalized' if X_comp is X_catch22_normalized else 'No Normalization'}")

    # Define model
    rf_classifier = RandomForestClassifier(random_state=RANDOM_STATE)
    
    # Setup for comparison
    mask = (y == class_comparison[0]) | (y == class_comparison[1])
    X_comp_filtered = X_comp[mask]
    y_comp_filtered = y[mask]
    group_labels = subjects[mask]
    y_binary = (y_comp_filtered == class_comparison[1]).astype(int)

    print(f"Training model: {rf_classifier}...")
    metrics = loso_binary_nested_cv(
        X_comp_filtered, 
        y_binary, 
        group_labels, 
        rf_classifier, 
        PARAM_GRID_RF, 
        'classifier'
        )
    
    # safe results
    result = safe_results_binary(
        model_name='RandomForest',
        personalization='Normalized' if X_comp is X_catch22_normalized else 'NoNormalization',
        metrics=metrics
    )
    comparison_results.append(result)

    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")


Personalization: No Normalization
Training model: RandomForestClassifier(random_state=42)...


    Accuracy : 0.895 ± 0.104
    F1       : 0.883 ± 0.134
    AUC      : 0.972 ± 0.053
Personalization: Baseline Normalized
Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.939 ± 0.073
    F1       : 0.934 ± 0.083
    AUC      : 0.985 ± 0.028


## Mit Kalibrierung + baseline norm - Random Forest Classifier

In [ ]:
# Define comparisons
class_comparison = (0, 5)

# Define model
rf_classifier = RandomForestClassifier(random_state=RANDOM_STATE)

# Define calibration sample size (k=4 means 2 samples per class for binary classification)
calibration_comparison = [2, 4, 6, 8]

for k in calibration_comparison:
    # Setup for comparison
    mask = (y == class_comparison[0]) | (y == class_comparison[1])
    X_comp_filtered = X_catch22_normalized[mask]
    y_comp_filtered = y[mask]
    group_labels = subjects[mask]
    y_binary = (y_comp_filtered == class_comparison[1]).astype(int)

    print (f"Calibration samples per class: {k//2} (total {k})")
    print(f"Training model: {rf_classifier}...")
    metrics = loso_binary_calibrated_nested_cv(
        X_comp_filtered, 
        y_binary, 
        group_labels, 
        rf_classifier, 
        PARAM_GRID_RF, 
        k
    )

    #safe results
    result = safe_results_binary(
        model_name='RandomForest',
        personalization=f'Calibrated_{k}',
        metrics=metrics
    )

    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")

Calibration samples per class: 1 (total 2)
Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.941 ± 0.080
    F1       : 0.934 ± 0.102
    AUC      : 0.987 ± 0.029
Calibration samples per class: 2 (total 4)
Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.937 ± 0.085
    F1       : 0.931 ± 0.102
    AUC      : 0.985 ± 0.035
Calibration samples per class: 3 (total 6)
Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.925 ± 0.097
    F1       : 0.915 ± 0.118
    AUC      : 0.984 ± 0.040
Calibration samples per class: 4 (total 8)
Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.962 ± 0.062
    F1       : 0.959 ± 0.070
    AUC      : 0.993 ± 0.029


Frage: Sind es genug samples zum kalibrieren? 

# Voll Personalisiert - Random Forest Classifier 

Frage: Sind die Daten genügend um damit ein voll personalisiertes Model zu bauen? 

# XGBoost

## Binary Klassifikation t0 vs t5 - XGBoost

In [ ]:
# Define comparison
class_comparison = (0, 5)
pers_comparison = [X_catch22, X_catch22_normalized]


# Train and evaluate
for X_comp in pers_comparison:

    print(f"Personalization: {'Baseline Normalized' if X_comp is X_catch22_normalized else 'No Normalization'}")

    xgb_classifier = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')
    
    # Setup for comparison
    mask = (y == class_comparison[0]) | (y == class_comparison[1])
    X_comp_filtered = X_comp[mask]
    y_comp_filtered = y[mask]
    group_labels = subjects[mask]
    
    y_binary = (y_comp_filtered == class_comparison[1]).astype(int)

    print(f"Training model: {xgb_classifier}...")
    
    metrics = loso_binary_nested_cv(
        X_comp_filtered, 
        y_binary, 
        group_labels, 
        xgb_classifier, 
        PARAM_GRID_XGB, 
        'classifier'
        )
    
    # safe results
    result = safe_results_binary(
        model_name='XGBoost',
        personalization='Normalized' if X_comp is X_catch22_normalized else 'NoNormalization',
        metrics=metrics
    )
    comparison_results.append(result)

    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")

## Mit kalibrierung + basline norm - XGBoost

In [ ]:
# Define comparisons
class_comparison = (0, 5)

# Define model
xgb_classifier = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')

# Define calibration sample size (k=4 means 2 samples per class for binary classification)
calibration_comparison = [2, 4, 6, 8]

for k in calibration_comparison:
    # Setup for comparison
    mask = (y == class_comparison[0]) | (y == class_comparison[1])
    X_comp_filtered = X_catch22_normalized[mask]
    y_comp_filtered = y[mask]
    group_labels = subjects[mask]
    y_binary = (y_comp_filtered == class_comparison[1]).astype(int)

    print (f"Calibration samples per class: {k//2} (total {k})")
    print(f"Training model: {xgb_classifier}...")
    metrics = loso_binary_calibrated_nested_cv(
        X_comp_filtered, 
        y_binary, 
        group_labels, 
        xgb_classifier, 
        PARAM_GRID_XGB, 
        k
    )

   # safe results
    result = safe_results_binary(
        model_name='XGBoost',
        personalization='Normalized' if X_comp is X_catch22_normalized else 'NoNormalization',
        metrics=metrics
    )
    comparison_results.append(result)

    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")

## Voll personalisiert

# Direkter vergleich - der Beiden Classifier

In [ ]:
df_all = pd.DataFrame(comparison_results)
display(df_all.round(3))

## 